In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

In [ ]:
projdir = '/u/project/cluo/terencew/claude/igvf/pgp/ipsc_le_se_rerun'
pbdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
pubdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv'
outdir = f'{projdir}/csv'
samples = list(np.loadtxt(f'{pbdir}/txt/allc_table/ips_cluster_donor.txt', dtype=str))
b_size = 200
w_size = 100000
n_windows = 500
tmp_slack = 200
samples

### PMDs with 100 kb flanks inside the chromosome, as in 09_ipsc_subtypes/17_pmds_metagene

In [ ]:
pmds = pd.read_csv(f'{pbdir}/pmds/dnmtools/rna_cluster_donor/merged/start_merged.filt_mcg.sorted.bed.gz', sep='\t', header=None)
pmds.index = [f'{x}_{y}' for x, y in zip(pmds[0], pmds[1])]
chrom_sizes = pd.read_csv('/u/project/cluo/terencew/reference/hg38_igvf/GRCh38.autosomal.chromsizes', sep='\t', header=None)
chrom_dict = dict(zip(chrom_sizes[0], chrom_sizes[1]))
tmp_sizes = [chrom_dict.get(x) for x in pmds[0]]
mask = (pmds[1] - w_size > 0) & (((pmds[2] + w_size) - tmp_sizes) < 0)
final_pmds = pmds[mask]
pmds.shape, final_pmds.shape

### 200 bp flanks from the sparse bins (zero-filled per chromosome; same sums as tabix on the published .full files)

In [ ]:
def load_bins(path):
    df = pd.read_csv(path, sep='\t', header=None, usecols=[0, 1, 4, 5])
    mc, cov = {}, {}
    for c, x in df.groupby(0):
        if c not in chrom_dict:
            continue
        m = np.zeros(chrom_dict[c] // b_size + 1, dtype=np.int64)
        v = np.zeros(chrom_dict[c] // b_size + 1, dtype=np.int64)
        m[x[1].values // b_size] = x[4].values
        v[x[1].values // b_size] = x[5].values
        mc[c], cov[c] = m, v
    return mc, cov

def flank_mcg(mc, cov, starts):
    k = np.arange(n_windows)
    tm = np.zeros(n_windows)
    tc = np.zeros(n_windows)
    for c, s in zip(final_pmds[0], starts):
        idx = s // b_size + k
        tm += mc[c][idx]
        tc += cov[c][idx]
    return pd.DataFrame(tm / tc)

def get_metagene(s, bins_dir, pmds_dir):
    mc, cov = load_bins(f'{bins_dir}/{s}.chrom{b_size}_CGN-Both.sparse.bed.gz')
    left_starts = [int(round(x / b_size)) * b_size - w_size for x in final_pmds[1]]
    right_starts = [int(round(x / b_size)) * b_size for x in final_pmds[2]]
    left_plot = flank_mcg(mc, cov, left_starts)
    right_plot = flank_mcg(mc, cov, right_starts)
    allc = pd.read_csv(f'{pmds_dir}/{s}.pmds_windows_CGN-Both.full.bed.gz', sep='\t', header=None)
    allc.index = [f'{x}_{y}' for x, y in zip(allc[0], allc[1])]
    indices = np.where(allc.index.isin(final_pmds.index))[0]
    tmp_mc = np.sum([allc.iloc[idx:idx + n_windows, :][3].values for idx in indices], axis=0)
    tmp_cov = np.sum([allc.iloc[idx:idx + n_windows, :][4].values for idx in indices], axis=0)
    pmd_plot = pd.DataFrame(tmp_mc / tmp_cov)
    final_plot = pd.concat([left_plot.iloc[-tmp_slack:], pmd_plot, right_plot.iloc[:tmp_slack]])
    final_plot['tmp'] = np.arange(final_plot.shape[0])
    final_plot['sample'] = s
    return final_plot

### check: published pseudobulks reproduce the published metagene

In [ ]:
pub = pd.read_csv(f'{pubdir}/figure5/ips_cluster_donor_pmds_metagene.csv', sep='\t', index_col=0)
s = 'IPS_main_C29'
check = get_metagene(s, f'{pbdir}/bed/bins/ips_cluster_donor', f'{pbdir}/bed/pmds/ips_cluster_donor')
a = check[0].values
b = pub[pub['sample'] == s]['0'].values
check.shape, np.abs(a - b).max(), np.corrcoef(a, b)[0, 1]

### early-iPSC pseudobulks

In [ ]:
def run(s):
    return get_metagene(s, f'{projdir}/bed/bins/ips_cluster_donor', f'{projdir}/bed/pmds/ips_cluster_donor')

with ProcessPoolExecutor(max_workers=4) as executor:
    pmd_meta_results = list(tqdm(executor.map(run, samples), total=len(samples)))
merged = pd.concat(pmd_meta_results)
merged['cluster'] = ['_'.join(x.split('_')[:2]) for x in merged['sample']]
merged['donor'] = [x.split('_')[2] for x in merged['sample']]
merged.head()

In [ ]:
merged.shape

In [ ]:
comp = merged[['sample', 'tmp', 0]].rename(columns={0: 'early'}).merge(pub[['sample', 'tmp', '0']].rename(columns={'0': 'pub'}), on=['sample', 'tmp'])
comp.groupby('sample').apply(lambda x: pd.Series({'r': np.corrcoef(x['early'], x['pub'])[0, 1], 'mean_early': x['early'].mean(), 'mean_pub': x['pub'].mean()}))

In [ ]:
merged.to_csv(f'{outdir}/ips_cluster_donor_pmds_metagene.csv', sep='\t')

In [ ]:
!date